# RAE JAX CelebA-HQ Kaggle Pipeline (moe1)

Notebook này bám flow Kaggle của repo:

- clone repo rồi checkout branch `jax-sit-dh-moe1-celebahq256`
- đồng bộ dependency từ `pyproject.toml` vào `/tmp/.venv` bằng `uv sync`
- chạy mọi bước cần package của repo qua `uv run` để không lệch khỏi env riêng
- tải `eurecom-ds/celeba-hq-256` từ Hugging Face rồi export thành `ImageFolder` thật sự ở kích thước `256x256`
- tạo `bootstrap_identity_stat.pt` để Stage 1 JAX có thể encode latent chưa chuẩn hoá
- tính `stage_1` latent normalization stats cho CelebA-HQ bằng `src_jax/build_stage1_stats.py`
- chạy Stage 1 single-image reconstruction và export recon folder bằng `src_jax/reconstruct_folder.py`
- build FID reference stats cho split validation
- build diagonal GMM artifact cho learned source `moe1` bằng `src_jax/build_source_gmm.py`
- ghi sẵn config Stage 2 CelebA-HQ với block `source.enabled=true` cho `src_jax/train.py`

Lưu ý:

- notebook dùng Hugging Face dataset `eurecom-ds/celeba-hq-256`, nên không còn phụ thuộc `manual_dir` hay bộ tar thủ công của TFDS
- cell lấy secret Kaggle và cell display vẫn dùng kernel Kaggle gốc
- các bước nặng phụ thuộc package như export dữ liệu, build stat, reconstruction, FID, và train đều đi qua môi trường `uv` tại `/tmp/.venv`
- notebook này ghim `transformers==4.57.1`; repo sẽ tự vá cache `diffuse_nnx` để import Dinov2 từ subpackage tương thích của `transformers` và lazy-load `google.cloud.storage` chỉ khi backend thực sự cần tải asset từ GCS

Giới hạn hiện tại:

- branch `jax-sit-dh-moe1-celebahq256` vẫn giữ pipeline train JAX hiện tại nhận `ImageFolder`; notebook chỉ đổi nguồn download sang Hugging Face rồi materialize dữ liệu đầu vào tại chỗ
- notebook này bao phủ Stage 1 inference/stat prep và Stage 2 prep/train command theo pipeline Kaggle


In [ ]:
%cd /kaggle/working
!rm -rf RAE
!git clone https://github.com/sontungkieu/RAE
%cd /kaggle/working/RAE
!git checkout jax-sit-dh-moe1-celebahq256
!curl -LsSf https://astral.sh/uv/install.sh | sh
!ln -sf /root/.local/bin/uv /usr/local/bin/uv


In [ ]:
import os

os.environ["UV_PROJECT_ENVIRONMENT"] = "/tmp/.venv"
os.environ["UV_CACHE_DIR"] = "/tmp/uv-cache"

!uv sync -q
print("Synced the repo dependencies into /tmp/.venv. The package-backed steps below all go through uv run.")


In [ ]:
import os
import pathlib

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["PYOPENGL_PLATFORM"] = "egl"

try:
    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()
    wandb_token = secrets.get_secret("WANDB2")
    hf_token = secrets.get_secret("HF_TOK_WRITE_KAGGLE")

    os.environ["WANDB_API_KEY"] = wandb_token
    os.environ["WANDB_KEY"] = wandb_token
    os.environ["HF_TOKEN"] = hf_token

    netrc = pathlib.Path.home() / ".netrc"
    netrc.write_text(f"machine api.wandb.ai login user password {wandb_token}\n")
    os.chmod(netrc, 0o600)
    print("Loaded Kaggle secrets for wandb and Hugging Face.")
except Exception as exc:
    print(f"Skipping Kaggle secret bootstrap: {exc}")


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

uv run hf download nyu-visionx/RAE-collections \
  decoders/dinov2/wReg_base/ViTXL_n08/model.pt \
  --local-dir models


In [ ]:
from pathlib import Path

repo_root = Path("/kaggle/working/RAE")
hf_cache_dir = Path("/kaggle/working/hf_datasets_cache")
celebahq_root = Path("/kaggle/working/celebahq256_imgfolder")
stage1_cfg_path = repo_root / "configs" / "stage1" / "pretrained" / "CelebAHQ256_DINOv2-B_jax.yaml"
stage2_cfg_path = repo_root / "configs" / "stage2" / "training" / "CelebAHQ256_SiTDH-S_DINOv2-B_moe1_jax.yaml"
bootstrap_stats_path = Path("/kaggle/working/bootstrap_identity_stat.pt")
latent_stats_path = Path("/kaggle/working/celebahq256_stage1_latent_stat.pt")
fid_stats_path = Path("/kaggle/working/celebahq256_val_fid_stats.pkl")
source_gmm_path = Path("/kaggle/working/celebahq256_source_gmm.npz")
stage1_single_recon_path = Path("/kaggle/working/celebahq256_stage1_single_recon.png")
stage1_recon_dir = Path("/kaggle/working/celebahq256_stage1_recon_val")
stage2_results_dir = Path("/kaggle/working/results_jax")

stats_batch_size = 16
stats_num_workers = 8
recon_batch_size = 8
recon_num_workers = 8
recon_limit = 2048  # đặt None hoặc bỏ --limit ở cell dưới nếu muốn export toàn bộ val split

print("repo_root:", repo_root)
print("hf_cache_dir:", hf_cache_dir)
print("celebahq_root:", celebahq_root)
print("stage1_cfg_path:", stage1_cfg_path)
print("stage2_cfg_path:", stage2_cfg_path)
print("source_gmm_path:", source_gmm_path)
print("All package-backed cells below use uv run against the synced /tmp/.venv environment.")


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

uv run python src_jax/export_celebahq_hf.py \
  --dataset eurecom-ds/celeba-hq-256 \
  --cache-dir /kaggle/working/hf_datasets_cache \
  --output /kaggle/working/celebahq256_imgfolder

uv run python - <<'PYSUM'
import json
from pathlib import Path

summary_path = Path("/kaggle/working/celebahq256_imgfolder/hf_export_summary.json")
summary = json.loads(summary_path.read_text())
print(json.dumps(summary, indent=2))
PYSUM


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

uv run python - <<'PY'
from pathlib import Path
repo_root = Path("/kaggle/working/RAE")
celebahq_root = Path("/kaggle/working/celebahq256_imgfolder")
stage1_cfg_path = repo_root / "configs" / "stage1" / "pretrained" / "CelebAHQ256_DINOv2-B_jax.yaml"
stage2_cfg_path = repo_root / "configs" / "stage2" / "training" / "CelebAHQ256_SiTDH-S_DINOv2-B_moe1_jax.yaml"
bootstrap_stats_path = Path("/kaggle/working/bootstrap_identity_stat.pt")
latent_stats_path = Path("/kaggle/working/celebahq256_stage1_latent_stat.pt")
fid_stats_path = Path("/kaggle/working/celebahq256_val_fid_stats.pkl")
source_gmm_path = Path("/kaggle/working/celebahq256_source_gmm.npz")

import textwrap

import torch

bootstrap_stats_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(
    {
        "mean": torch.zeros((1, 1, 1), dtype=torch.float32),
        "var": torch.ones((1, 1, 1), dtype=torch.float32),
        "count": 0,
    },
    bootstrap_stats_path,
)

stage1_cfg_text = textwrap.dedent(
    f"""
    stage_1:
      target: stage1.RAE
      params:
        encoder_cls: 'Dinov2withNorm'
        encoder_config_path: 'facebook/dinov2-with-registers-base'
        encoder_input_size: 224
        encoder_params:
          dinov2_path: 'facebook/dinov2-with-registers-base'
          normalize: true
        decoder_config_path: 'configs/decoder/ViTXL'
        pretrained_decoder_path: 'models/decoders/dinov2/wReg_base/ViTXL_n08/model.pt'
        noise_tau: 0.0
        reshape_to_2d: true
        normalization_stat_path: '{latent_stats_path.as_posix()}'
    """
).strip() + "\n"

stage2_cfg_text = textwrap.dedent(
    f"""
    stage_1:
      target: stage1.RAE
      ckpt: null
      params:
        encoder_cls: 'Dinov2withNorm'
        encoder_config_path: 'facebook/dinov2-with-registers-base'
        encoder_input_size: 224
        encoder_params:
          dinov2_path: 'facebook/dinov2-with-registers-base'
          normalize: true
        decoder_config_path: 'configs/decoder/ViTXL'
        pretrained_decoder_path: 'models/decoders/dinov2/wReg_base/ViTXL_n08/model.pt'
        noise_tau: 0.0
        reshape_to_2d: true
        normalization_stat_path: '{latent_stats_path.as_posix()}'

    stage_2:
      target: stage2.models.SiT.SiTDH
      ckpt: null
      params:
        input_size: 16
        patch_size: 1
        in_channels: 768
        hidden_size: [384, 2048]
        depth: [12, 2]
        num_heads: [6, 16]
        mlp_ratio: 4.0
        class_dropout_prob: 0.0
        num_classes: 1
        use_qknorm: false
        use_swiglu: true
        use_rope: true
        use_rmsnorm: true
        use_pos_embed: true
        wo_shift: false

    transport:
      params:
        path_type: 'Linear'
        prediction: 'velocity'
        loss_weight: null
        time_dist_type: 'uniform'

    sampler:
      mode: ODE
      params:
        sampling_method: 'euler'
        num_steps: 50
        atol: 1.0e-6
        rtol: 1.0e-3
        reverse: false

    guidance:
      method: 'cfg'
      scale: 1.0
      t_min: 0.0
      t_max: 1.0

    misc:
      latent_size: [768, 16, 16]
      num_classes: 1
      time_dist_shift_dim: 196608
      time_dist_shift_base: 4096

    eval:
      data_path: '{(celebahq_root / "val").as_posix()}'
      eval_every: 5000
      batch_size: 4
      num_workers: 0
      max_batches: 32
      eval_model: false
      fid_ref: '{fid_stats_path.as_posix()}'
      fid_every: 5000
      fid_num_samples: 4096
      fid_per_proc_batch_size: 4
      fid_batch_size: 128


    source:
      enabled: true
      kind: gmm_moe1
      gmm_stats_path: '{source_gmm_path.as_posix()}'
      num_modes: 4
      condition_dim: 16
      hidden_channels: 256
      router_temperature: 2.0
      soft_moe: true
      balance_loss_weight: 0.1
      entropy_loss_weight: 1.0e-2
      var_kl_loss_weight: 1.0
      target_variance: 1.0
      logvar_min: -8.0
      logvar_max: 4.0
      var_floor: 1.0e-5
      posterior_eps: 1.0e-6
      weight_prior: 0.01
      em_iters: 100
      em_tol: 1.0e-4
      em_restarts: 3
      dead_count_threshold: 1.0
      active_mode_fraction_threshold: 0.01

    training:
      global_seed: 0
      epochs: 200
      global_batch_size: 64
      grad_accum_steps: 2
      ema_decay: 0.9995
      num_workers: 0
      random_flip: true
      log_every: 10
      ckpt_every: 5000
      sample_every: 5000
      base_lr: 0.0001
      final_lr: 0.00001
      beta: [0.9, 0.95]
      wd: 0.0
      schedule_type: 'linear'
      decay_start_epoch: 150
      decay_end_epoch: 200
      clip_grad: 1.0
    """
).strip() + "\n"

stage1_cfg_path.parent.mkdir(parents=True, exist_ok=True)
stage2_cfg_path.parent.mkdir(parents=True, exist_ok=True)
stage1_cfg_path.write_text(stage1_cfg_text, encoding="utf-8")
stage2_cfg_path.write_text(stage2_cfg_text, encoding="utf-8")

print(f"Wrote {stage1_cfg_path}")
print(f"Wrote {stage2_cfg_path}")
print(f"Bootstrap stats path: {bootstrap_stats_path}")
PY


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

uv run python src_jax/build_stage1_stats.py \
  --config configs/stage1/pretrained/CelebAHQ256_DINOv2-B_jax.yaml \
  --input /kaggle/working/celebahq256_imgfolder/train \
  --output /kaggle/working/celebahq256_stage1_latent_stat.pt \
  --batch-size 16 \
  --num-workers 8 \
  --set stage_1.params.normalization_stat_path=/kaggle/working/bootstrap_identity_stat.pt


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

uv run python - <<'PY'
from pathlib import Path
latent_stats_path = Path("/kaggle/working/celebahq256_stage1_latent_stat.pt")
celebahq_root = Path("/kaggle/working/celebahq256_imgfolder")

import torch

stats = torch.load(latent_stats_path, map_location="cpu")
print("latent mean shape:", tuple(stats["mean"].shape))
print("latent var shape:", tuple(stats["var"].shape))
print("num samples:", stats["count"])

sample_image_path = next(path for path in (celebahq_root / "val" / "face").iterdir() if path.is_file())
print("sample image:", sample_image_path)
PY


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

sample_image=$(find /kaggle/working/celebahq256_imgfolder/val/face -type f -print -quit)
[ -n "${sample_image}" ]

uv run python src_jax/stage1_sample.py \
  --config configs/stage1/pretrained/CelebAHQ256_DINOv2-B_jax.yaml \
  --image "${sample_image}" \
  --output /kaggle/working/celebahq256_stage1_single_recon.png


In [ ]:
display(DisplayImage(filename=stage1_single_recon_path))


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

uv run python src_jax/reconstruct_folder.py \
  --config configs/stage1/pretrained/CelebAHQ256_DINOv2-B_jax.yaml \
  --input /kaggle/working/celebahq256_imgfolder/val \
  --output-dir /kaggle/working/celebahq256_stage1_recon_val \
  --batch-size 8 \
  --num-workers 8 \
  --limit 2048


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

uv run python src_jax/build_fid_stats.py \
  --input /kaggle/working/celebahq256_imgfolder/val \
  --output /kaggle/working/celebahq256_val_fid_stats.pkl \
  --image-size 256 \
  --batch-size 64 \
  --num-workers 32


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

uv run python src_jax/build_source_gmm.py \
  --config configs/stage1/pretrained/CelebAHQ256_DINOv2-B_jax.yaml \
  --input /kaggle/working/celebahq256_imgfolder/train \
  --output /kaggle/working/celebahq256_source_gmm.npz \
  --batch-size 16 \
  --num-workers 8 \
  --num-modes 4 \
  --chunk-size 128


## Optional: launch Stage 2 JAX training on Kaggle GPU

Cell dưới đây dùng config CelebA-HQ vừa ghi ở trên. Nếu bạn chỉ cần Stage 1 + stats thì có thể dừng ở đây.

Cell bên trên build artifact diagonal GMM cho learned source `moe1` trước khi train Stage 2.


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

export ENTITY="<wandb_entity>"
export PROJECT="rae-jax-celebahq256-moe1-$(TZ=Asia/Bangkok date +%Y%m%d-%H%M%S)"

export RAE_JAX_REBUILD_BACKEND=1

uv run python src_jax/train.py \
  --config configs/stage2/training/CelebAHQ256_SiTDH-S_DINOv2-B_moe1_jax.yaml \
  --data-path /kaggle/working/celebahq256_imgfolder \
  --results-dir /kaggle/working/results_jax \
  --precision bf16 \
  --set training.log_rae_latent_stats=true \
  --set training.log_activation_stats=true \
  --wandb
